# 02 — Cleaning pipline

## Setup

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))
from load import load_cgm_file, load_summary_file, cgm_files, summary_files
import pandas as pd


pd.set_option("display.max_columns", None)
pd.set_option("display.width", 400)

table_a = pd.concat([load_cgm_file(f) for f in cgm_files], ignore_index=True)
table_b = pd.concat([load_summary_file(f) for f in summary_files], ignore_index=True)

print(type(table_a))
print("table b shape: ", table_b.shape)

print(table_a.head)
#print(table_b.info())

<class 'pandas.DataFrame'>
table b shape:  (125, 37)
<bound method NDFrame.head of         subject  visit recording_date                date  cgm_mg_dl  cbg_mg_dl  blood_ketone_mmol_l      dietary_intake dietary_intake_zh insulin_dose_sc non_insulin_hypoglycemic_agents insulin_csii_bolus_r_iu insulin_csii_basal_r_iu_h insulin_dose_iv  source_row           source_file
0          1001      0     2021-07-30 2021-07-30 16:43:00      113.4        NaN                  NaN                 NaN               NaN             NaN                             NaN                     NaN                       0.3             NaN           0  1001_0_20210730.xlsx
1          1001      0     2021-07-30 2021-07-30 16:58:00      124.2        NaN                  NaN                 NaN               NaN             NaN                             NaN                     4.0                       NaN             NaN           1  1001_0_20210730.xlsx
2          1001      0     2021-07-30 2021-07-30 17:13:0

## Data dictionary

**Table A —— long CGM readings**
Grain: one row per glucose reading. Roughly 128,000 rows across the cohort.
(drop: '饮食' because it's dulplicat from 'Dietary intake')

Columns (14): 
'subject_id'
'visit'
'date'
'cgm_mg_dl'
'cbg_mg_dl'
'blood_ketone_mmol_l'
'dietary_intake'
'insulin_dose_sc'
'non-insulin_hypoglycemic_agents'
'csii_bolus_insulin_novolin_r_iu'
'csii_basal_insulin_novolin_r_iu_h'
'insulin_dose_iv'
'source_file'
'source_row'

**Table B —— Clinical summary**
Grain: one row per patient-visit. 125 rows.
Columns:
'subject id'
'gender (female=1, male=2)'
'age (years)'
'height (m)'
'weight (kg)'
'bmi (kg/m2)'
'smoking history (pack year)'
'alcohol drinking history (drinker/non-drinker)'
'type of diabetesduration of diabetes  (years)'
'acute diabetic complications'
'diabetic macrovascular  complications'
'diabetic microvascular complications'
'comorbidities'
'hypoglycemic agents'
'other agents'
'fasting plasma glucose (mg/dl)'
'2-hour postprandial plasma glucose (mg/dl)'
'fasting c-peptide (nmol/l)'
'2-hour postprandial c-peptide (nmol/l)'
'fasting insulin (pmol/l)'
'2-hour postprandial insulin (pmol/l)'
'hba1c (mmol/mol)'
'glycated albumin (%)'
'total cholesterol (mmol/l)'
'triglyceride (mmol/l)'
'high-density lipoprotein cholesterol (mmol/l)'
'low-density lipoprotein cholesterol (mmol/l)'
'creatinine (umol/l)'
'estimated glomerular filtration rate  (ml/min/1.73m2)' 
'uric acid (mmol/l)'
'blood urea nitrogen (mmol/l)'
'hypoglycemia (yes/no)'

## 1. Structural integrity

Before any values get touched: confirm the two tables actually join.
Verified: table_a and table_b share identical subject+visit keys, no orphans either direction.

In [2]:
#merge table_a and table_b
left = table_a[["subject", "visit"]]
right = table_b[["subject", "visit"]]
merged = left.merge(right, on=["subject", "visit"], how="outer", indicator=True)
#print(merged)
mismatches = merged[merged["_merge"] != "both"]
print(mismatches)

#check one row per subject+visit with no duplicates
print(table_b.duplicated(subset=["subject", "visit"]).sum())

Empty DataFrame
Columns: [subject, visit, _merge]
Index: []
0


## 2. Table A cleaning

Applies `clean_cgm` from `src/clean.py`: drops the duplicate `dietary_intake_zh` column (D-009), rounds timestamps to the nearest minute (D-014), resamples each patient-visit onto a regular 15-minute grid — restarting the origin at any real device-swap gap rather than a single grid per visit (D-015) — flags missing glucose readings (`cgm_gap`, D-012/D-017) and out-of-range CGM/CBG values (D-010) without imputing anything.


In [3]:
from clean import clean_cgm

table_a_clean = clean_cgm(table_a)

print("rows:", len(table_a_clean))
print("cgm_gap flagged:", table_a_clean["cgm_gap"].sum())
print("cgm_out_of_range flagged:", table_a_clean["cgm_out_of_range"].sum())
print("cbg_out_of_range flagged:", table_a_clean["cbg_out_of_range"].sum())

rows: 128185
cgm_gap flagged: 28
cgm_out_of_range flagged: 0
cbg_out_of_range flagged: 1


## 3. Table B cleaning

Check lab and anthropometric columns for implausible values. Check categorical columns (diabetes_type, comorbidities, alcohol_history, has_hypoglycemia, etc.) for consistent spelling/casing. Missing lab values (HbA1c, insulin, C-peptide) stay NaN — documented missingness, not imputed.

In [4]:
from clean import clean_summary

table_b_clean = clean_summary(table_b)

print("shape:", table_b_clean.shape)
print("uric_acid_umol_l" in table_b_clean.columns)

shape: (125, 37)
True


## 4. Write cleaned tables

Save cleaned Table A and Table B to data/interim/, so downstream notebooks load the cleaned version instead of re-running this pipeline each time.

In [5]:
table_a_clean.to_parquet("../data/interim/table_a_clean.parquet", index=False)
table_b_clean.to_parquet("../data/interim/table_b_clean.parquet", index=False)

In [6]:
print(pd.read_parquet("../data/interim/table_a_clean.parquet").head())


   subject  visit  segment                date recording_date  cgm_mg_dl  cbg_mg_dl  blood_ketone_mmol_l      dietary_intake insulin_dose_sc non_insulin_hypoglycemic_agents  insulin_csii_bolus_r_iu  insulin_csii_basal_r_iu_h insulin_dose_iv  source_row           source_file  cgm_gap  cgm_out_of_range  cbg_out_of_range
0     1001      0        0 2021-07-30 16:43:00     2021-07-30      113.4        NaN                  NaN                 NaN             NaN                             NaN                      NaN                        0.3             NaN         0.0  1001_0_20210730.xlsx    False             False             False
1     1001      0        0 2021-07-30 16:58:00     2021-07-30      124.2        NaN                  NaN                 NaN             NaN                             NaN                      4.0                        NaN             NaN         1.0  1001_0_20210730.xlsx    False             False             False
2     1001      0        0 2021-07-30 17

## 5. Cohort and attrition check

Confirm no subjects, visits, or rows are silently dropped anywhere between the
raw files and the cleaned tables.


| Stage | Count |
|---|---|
|CGM files loaded|125|
|Summary files loaded|2 (Shanghai_T1DM_Summary.xlsx, Shanghai_T2DM_Summary.xlsx)|
|Distinct subjects|112 (100 T2DM + 12 T1DM — matches the plan's estimate exactly)|
|Subject-visits|125 (102 subjects × 1 visit, 7 × 2, 3 × 3 — D-001)|
|table_a raw concat rows|128,170|
|table_a_clean rows|128,185|
|table_b raw concat rows|125|
|table_b_clean rows|125|


In [7]:
print("CGM files loaded:", len(cgm_files))
print("Summary files loaded:", len(summary_files))
print()
print("Distinct subjects (table_a):", table_a["subject"].nunique())
print("Subject-visits (table_a):", table_a[["subject", "visit"]].drop_duplicates().shape[0])
print()
print("table_a raw rows:  ", len(table_a))
print("table_a_clean rows:", len(table_a_clean))
print()
print("table_b raw rows:  ", len(table_b))
print("table_b_clean rows:", len(table_b_clean))


CGM files loaded: 125
Summary files loaded: 2

Distinct subjects (table_a): 112
Subject-visits (table_a): 125

table_a raw rows:   128170
table_a_clean rows: 128185

table_b raw rows:   125
table_b_clean rows: 125
